# 03.02 环境准备、模型下载与数据预处理

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 03.01 章节概述</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">创建昇腾 NPU 环境，下载模型，完成数据预处理</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">环境与依赖安装 → 模型下载 → 数据预处理（对话模板+掩码）</td></tr>
</table>

## 第一部分：昇腾 NPU 环境准备


## 1. 为什么需要昇腾 NPU  
大语言模型微调对算力和显存要求很高。本实验使用 **DeepSeek-R1-Distill-Qwen-1.5B**（约 17.86 亿参数），模型权重约 3.3GB。在 CPU 上训练一条样本就要几秒，而昇腾 NPU 可以把整个训练流程从数小时压缩到数十分钟。

## 2. 安装 Python 依赖

CANNLab 镜像已预装 **PyTorch + torch_npu**（让 PyTorch 识别昇腾 NPU），本节只需安装几个 HuggingFace 生态的依赖库（transformers / peft / datasets 等）。

> 💡 **为什么用 PyTorch 而不是 MindSpore？**
> CANNLab 环境默认提供 PyTorch + torch_npu，开箱即用、与 HuggingFace 生态完全兼容。MindSpore 在该环境的版本配套较复杂（pip 上的 mindspore 不支持 CANN 8.5），因此本课程采用 PyTorch 技术栈，学习者无需额外配置框架。

In [ ]:
# 配置清华 pip 源（加速国内下载）
!pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple

# 安装核心依赖（CANNLab 已预装 torch + torch_npu，无需再装框架）
# transformers：HuggingFace 模型加载与训练（PyTorch 后端）
!pip install transformers==4.55.4 -q

# peft：LoRA 等参数高效微调方法
!pip install peft==0.17.1 -q

# accelerate：分布式/混合精度训练加速
!pip install accelerate==1.10.1 -q

# 模型下载工具（从 ModelScope 下载模型）
!pip install modelscope -q

# datasets 库（数据预处理用）
!pip install datasets -q


### 各依赖库的作用

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">库</th><th align="left">作用</th></tr>
<tr><td align="left"><code>transformers</code></td><td align="left">加载 HuggingFace 模型、分词器，提供 <code>Trainer</code> 训练封装</td></tr>
<tr><td align="left"><code>peft</code></td><td align="left">LoRA 等参数高效微调方法</td></tr>
<tr><td align="left"><code>datasets</code></td><td align="left">数据加载与预处理（<code>Dataset.map</code> 批量处理）</td></tr>
<tr><td align="left"><code>accelerate</code></td><td align="left">分布式/混合精度训练加速</td></tr>
<tr><td align="left"><code>modelscope</code></td><td align="left">从魔搭社区下载模型（国内高速）</td></tr>
</table>

> ⚠️ **重要**：依赖安装完成后，请在菜单栏 **Kernel → Restart Kernel…** 重启内核，再运行下方验证代码。
> 不重启会导致新安装的 <code>transformers</code>、<code>peft</code> 无法被正确 import。

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">依赖</th><th align="left">作用</th></tr>
<tr><td align="left"><code>torch_npu</code></td><td align="left">让 PyTorch 识别并调用昇腾 NPU（CANNLab 已预装）</td></tr>
<tr><td align="left"><code>transformers==4.55.4</code></td><td align="left">模型结构、tokenizer、Trainer 等核心 API</td></tr>
<tr><td align="left"><code>peft==0.17.1</code></td><td align="left">LoRA 微调实现</td></tr>
<tr><td align="left"><code>accelerate==1.10.1</code></td><td align="left">设备分发（device_map）与分布式支持</td></tr>
<tr><td align="left"><code>modelscope</code></td><td align="left">从魔搭社区下载模型权重</td></tr>
</table>

In [ ]:
import torch
import torch_npu  # noqa: F401  让 PyTorch 识别昇腾 NPU
from datasets import Dataset
import pandas as pd
import transformers, peft
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          DataCollatorForSeq2Seq, TrainingArguments, Trainer,
                          GenerationConfig)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

# 环境自检：确认关键库版本 + NPU 可用性
print("PyTorch     :", torch.__version__)
print("torch_npu   :", torch_npu.__version__)
print("transformers:", transformers.__version__)
print("peft        :", peft.__version__)

# 确认 NPU 可用
if torch.npu.is_available():
    print(f"✅ 昇腾 NPU 可用: {torch.npu.get_device_name(0)}")
else:
    print("⚠️ 未检测到 NPU，将回退 CPU（训练会非常慢）")
print("✅ 环境就绪，可进入下一小节")


若上方打印的 PyTorch / torch_npu / transformers / peft 版本正常，且显示 <code>✅ 昇腾 NPU 可用</code>，说明环境准备完成。

> **常见问题**：若提示 "NPU 不可用" 或回退 CPU，请确认：
> - 已按 03.02 配置好 CANN 环境变量（<code>ASCEND_TOOLKIT_HOME</code>）
> - 已注册 Python 3.11.4 (CANN) 内核（详见课程前置说明）
> - 实例已开机（关机状态下 NPU 不可用）

---

## 本节练习

**练习 1（选择）**：为什么大模型微调必须用昇腾 NPU（或 GPU），而不能用 CPU？
- A. CPU 不支持 Python
- B. 大模型微调对显存和算力要求极高，CPU 远远无法满足
- C. NPU 比 CPU 便宜
- D. CPU 不能运行 MindSpore

**练习 2（填空）**：本节安装的三个核心依赖库及其版本分别是 ______ 、 ______ 、 ______。

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/03.02_env_dataset/answers_env.txt



---

## 第二部分：模型下载与数据预处理

### 1. 下载 DeepSeek-R1-Distill-Qwen-1.5B 模型

大模型文件体积较大（约 **3.3GB**），无法随课程仓库直接分发。本课程采用**自动下载**机制：

- 运行下方代码会**自动检测**本地是否已有模型
- 若无，则从 [魔搭 ModelScope](https://modelscope.cn)（国内高速镜像）下载
- 首次下载约 **5-15 分钟**（取决于网速），下载后会缓存在 `./src/DeepSeek-R1-Distill-Qwen-1.5B/`
- 后续运行会自动跳过，无需重复下载

> 💡 **为什么用 ModelScope 而不是 HuggingFace？**
> ModelScope 是国内的模型托管平台，国内网络访问速度快、稳定，无需翻墙和登录 token。

In [ ]:
# ===== 智能下载 DeepSeek-R1-Distill-Qwen-1.5B =====
# 自动检测：本地有模型就跳过，没有就从 ModelScope 下载
import os

MODEL_DIR = './src/DeepSeek-R1-Distill-Qwen-1.5B'
MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'

# 检测本地是否已有完整模型（以 model.safetensors 为标志）
has_model = os.path.exists(os.path.join(MODEL_DIR, 'model.safetensors'))

if has_model:
    size_gb = os.path.getsize(os.path.join(MODEL_DIR, 'model.safetensors')) / (1024**3)
    print(f"✅ 本地已有模型: {MODEL_DIR}")
    print(f"   model.safetensors: {size_gb:.2f} GB，跳过下载")
else:
    print(f"⏳ 本地未检测到模型，开始从 ModelScope 下载...")
    print(f"   模型: {MODEL_ID}")
    print(f"   目标: {MODEL_DIR}")
    print(f"   体积约 3.3GB，首次下载需 5-15 分钟，请耐心等待")
    print("-" * 60)
    # 安装 modelscope（如未安装）
    import subprocess
    try:
        import modelscope
    except ImportError:
        print("安装 modelscope...")
        subprocess.check_call(['pip', 'install', 'modelscope',
                             '-i', 'https://pypi.tuna.tsinghua.edu.cn/simple', '-q'])
    # 下载模型（完整下载所有文件）
    from modelscope import snapshot_download
    snapshot_download(MODEL_ID, local_dir=MODEL_DIR)
    print("-" * 60)
    print(f"✅ 下载完成！模型已保存到 {MODEL_DIR}")

# 列出模型文件，确认完整
print("\n模型文件清单:")
for fname in sorted(os.listdir(MODEL_DIR)):
    fpath = os.path.join(MODEL_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / (1024**2)
        print(f"  {fname:40s} {size_mb:8.2f} MB")

## 2. 下载并加载数据集

本实验使用 **Datawhale self-llm 甄嬛角色扮演数据集**（`huanhuan.json`，共 3729 条对话）。每条数据包含三部分：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">字段</th><th align="left">含义</th><th align="left">示例</th></tr>
<tr><td align="left"><code>instruction</code></td><td align="left">用户指令（必填）</td><td align="left">"小姐，您今年的生辰想要怎么过？"</td></tr>
<tr><td align="left"><code>input</code></td><td align="left">补充输入（可空）</td><td align="left">""</td></tr>
<tr><td align="left"><code>output</code></td><td align="left">期望的回答</td><td align="left">"今年的生辰，我希望……"</td></tr>
</table>

先下载数据集：

In [ ]:
# 下载数据集（约 1MB，秒下）
# huanhuan.json 已随课程提供在 ./src/，无需下载
# !wget "https://gh-proxy.com/https://raw.githubusercontent.com/datawhalechina/self-llm/refs/heads/master/dataset/huanhuan.json" -O huanhuan.json --no-check-certificate


接着用 `pandas` 读取 JSON 文件，再用 `datasets.Dataset.from_pandas` 转换为 HuggingFace 的 Dataset 对象（后者方便后续的批量 `map` 处理）：

In [ ]:
# 数据加载并进行格式转换
df = pd.read_json('./src/huanhuan.json')
ds = Dataset.from_pandas(df)

# 查看前 3 条原始数据
ds[:3]


## 3. 实例化分词器

**分词器（Tokenizer）** 的作用是把人类可读的文本转换成模型可处理的数字序列（token id）。它由两部分组成：

- **分词规则**：把句子切成子词（subword），例如"甄嬛"可能被切成 2-3 个 token；
- **词表**：每个子词对应一个数字编号。

我们用基础模型目录下的 tokenizer：

In [ ]:
# 实例化 tokenizer
# use_fast=False：使用 Python 实现的慢速分词器（昇腾环境兼容性更好）
# trust_remote_code=True：信任模型仓库的自定义代码
tokenizer = AutoTokenizer.from_pretrained('./src/DeepSeek-R1-Distill-Qwen-1.5B',
                                          use_fast=False,
                                          trust_remote_code=True)
tokenizer


## 4. 数据预处理：构造对话模板与标签（关键步骤）

这是微调中最核心的一步。我们要把每条 `{instruction, input, output}` 数据格式化成模型能学习的样本。

### 4.1 为什么要构造对话模板

不同模型有不同的对话模板，必须用**模型自带的模板**格式化训练数据，训练和推理才能对齐。本实验用的 **DeepSeek-R1-Distill-Qwen-1.5B** 继承了 DeepSeek 的模板，使用以下特殊 token（注意是全角 `｜`，不是 ChatML 的 `<|im_start|>`）：

```
<｜begin▁of▁sentence｜>[系统设定]<｜User｜>[用户输入]<｜Assistant｜>[助手回复]<｜end▁of▁sentence｜>
```

> ⚠️ **重要**：DeepSeek-R1-Distill 系列的词表里**没有** `<|im_start|>` / `<|im_end|>`（ChatML token），如果误用 ChatML，这些标记会被 BPE 拆成多个普通子词，模型学不到正确的角色边界，微调会失效。所以本节用 tokenizer 自带的 `apply_chat_template` 生成 prompt，确保与模型原生模板一致。

只有按这个格式喂给模型，训练出的模型才能在推理时正确续写回答。

### 4.2 labels 的设计：让模型只学"回答"不学"提问"

这是微调的关键技巧：**用 `-100` 标记不需要学习的位置**。

- `instruction`（system + user）部分 → labels 全部设为 `-100`（损失函数会忽略，模型不学这部分）；
- `response`（assistant）部分 → labels 保持原 token id（模型专注学习这部分）；
- 末尾补一个 `pad_token_id` 作为结束符（EOS），labels 也对应保留。

这样模型只学"如何回答"，而不是"如何提问"。

In [ ]:
def process_func(example):
    MAX_LENGTH = 384    # 分词器会把一个中文字切分为多个 token，适当放大长度保证完整性

    # ===== 用模型自带的 apply_chat_template 生成 prompt（与推理完全一致）=====
    # add_generation_prompt=True 会在末尾追加 "<｜Assistant｜>"，让模型从这里开始生成
    # 这样训练数据的 prompt 部分和 03.03 推理时的输入完全相同，避免训练/推理分布不一致
    instruction = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "现在你要扮演皇帝身边的女人--甄嬛"},
            {"role": "user", "content": example['instruction'] + example['input']},
        ],
        add_generation_prompt=True,   # 追加 <｜Assistant｜> 引导模型开始回答
        tokenize=True,
        return_dict=True,             # 返回 dict（含 input_ids/attention_mask）
    )

    # assistant 回复部分（这是模型需要学习的内容）
    response = tokenizer(example['output'], add_special_tokens=False)

    # 拼接成完整序列：[prompt 到 <｜Assistant｜>] + [回复] + [EOS 结束符]
    # pad_token_id == eos_token_id == <｜end▁of▁sentence｜>，作为回答结束标记
    input_ids = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
    # attention_mask：1 表示模型需要关注该位置，EOS 也要关注所以补 1
    attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
    # labels：prompt 部分用 -100 屏蔽（不学习），回复部分保留原 id（学习），末尾 EOS 保留
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]

    # 超长序列截断（防止超过模型最大长度）
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

### 4.3 对整个数据集批量处理

`Dataset.map` 会把 `process_func` 应用到每一条数据上。`remove_columns` 删掉原始的 `system/instruction/output` 列，只保留处理后的 `input_ids/attention_mask/labels`：

In [ ]:
# 对整个数据集应用预处理函数
tokenized_id = ds.map(process_func, remove_columns=ds.column_names)
tokenized_id


验证一下处理结果——把第一条样本的 `input_ids` 解码回文本，确认模板拼对了：

In [ ]:
# 解码查看第一条样本的完整内容（验证模板拼接是否正确）
tokenizer.decode(tokenized_id[0]['input_ids'])


你应该看到一段包含 `<｜begin▁of▁sentence｜>`（开头）、`<｜User｜>`（用户）、`<｜Assistant｜>`（助手）的完整对话文本，说明 DeepSeek 原生模板预处理正确。

数据预处理完成后，进入 [03.03 LoRA 配置、训练与推理](./03.03_lora_train_infer.ipynb)。

---

## 本节练习

**练习 1（选择）**：`labels` 中 `-100` 的作用是什么？
- A. 表示该位置是 padding
- B. 让损失函数忽略该位置，模型不学习这部分
- C. 表示该 token 出现错误
- D. 加速训练

**练习 2（填空）**：DeepSeek-R1-Distill-Qwen 模型用特殊 token ______ 标记用户发言、______ 标记助手发言，回答以 ______ 作为结束符。

**练习 3（代码）**：本实验把 system prompt 硬编码为"现在你要扮演皇帝身边的女人--甄嬛"。如果把角色换成"你现在是一位 Python 编程助手"，需要修改 `process_func` 中的哪一行？请写出修改后的代码。

> 💡 参考答案见下方 code cell。


In [ ]:
# 查看本节练习答案
!cat ./answer/03.02_env_dataset/answers_data.txt
